# Image Classification with Hugging Face on Google Colab

**Secure computer vision with 100% data privacy**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## Overview

Learn how to classify images using state-of-the-art vision transformers with complete data privacy.

---


## Step 1: Configure Security Environment

In [ ]:
import os

# Security configuration
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = "/content/.hf_cache"

!mkdir -p /content/.hf_cache

print("\u2705 Secure environment configured!")

## Step 2: Install Dependencies

In [ ]:
# Install vision packages
!pip install -q transformers datasets accelerate torch torchvision pillow

print("\u2705 Dependencies installed!")

## Step 3: Load Vision Model

We'll use ResNet-50, a well-tested model for image classification.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import requests

# Load model and processor
model_name = "microsoft/resnet-50"

print(f"Loading {model_name}...")

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

print(f"\u2705 Model loaded successfully!")
print(f"   Labels: {len(processor.id2label)} categories")

## Step 4: Create Secure Classification Function

In [ ]:
def classify_image_securely(image, top_k=5):
    """
    Classify image with memory cleanup.
    
    Args:
        image: PIL Image or URL
        top_k: Number of top predictions to return
    """
    # Load image from URL if string
    if isinstance(image, str):
        if image.startswith('http'):
            image = Image.open(requests.get(image, stream=True).raw)
        else:
            image = Image.open(image)
    
    # Preprocess
    inputs = processor(images=image, return_tensors="pt")
    
    # Classify with no gradients
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predictions
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits[0], dim=-1)
    top_probs, top_indices = probs.topk(top_k)
    
    # Clean up
    del inputs, outputs, logits
    
    # Format results
    results = []
    for prob, idx in zip(top_probs, top_indices):
        label = processor.id2label[idx.item()]
        results.append({
            "label": label,
            "confidence": f"{prob.item()*100:.2f}%"
        })
    
    return results

# Import torch for cleanup
import torch

print("\u2705 Classification function defined!")

## Step 5: Test with Sample Images

Let's test with some example images from the internet.

In [ ]:
# Test image URL (public domain image)
test_image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/be/Border_Collie_-_Aristotle.jpg/1200px-Border_Collie_-_Aristotle.jpg"

print(f"Testing with: {test_image_url}\n")
print("Classifying...")

results = classify_image_securely(test_image_url, top_k=5)

print("\nTop 5 Predictions:")
print("-" * 40)
for i, result in enumerate(results, 1):
    print(f"{i}. {result['label']}: {result['confidence']}")

## Step 6: Memory Cleanup

In [ ]:
import gc

def secure_cleanup():
    """Securely clear memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\u2705 Memory cleared securely!")

secure_cleanup()

---

## Security Notes

### Privacy Features

| Feature | Description |
|---------|-------------|
| **Image Processing** | All images processed in Colab VM memory |
| **No External API** | No cloud vision API calls |
| **Secure Cleanup** | Images deleted from memory after processing |
| **Telemetry Disabled** | No usage data sent anywhere |

### Image Privacy Best Practices

1. **Don't upload to external services** - Process locally
2. **Clear memory after use** - Call secure_cleanup()
3. **Don't save images to Drive** - Unless absolutely necessary
4. **Use URLs over uploads** - URLs load faster and use less memory

---

## Popular Vision Models

| Model | Task | Size |
|-------|------|------|
| **microsoft/resnet-50** | Classification | 98MB |
| **facebook/dinov2-base** | Feature extraction | 334MB |
| **openai/clip-vit-base-patch32** | Zero-shot classification | 151MB |
| **google/vit-base-patch16-224** | Vision Transformer | 86MB |

---

**Next**: Try the [Text Classification Notebook](./text_classification.ipynb)!

**Found this helpful?** Star the [GitHub repo](https://github.com/unn-Known1/huggingface-colab-secure-setup)!